In [2]:
"""
Bike Rebalancing with OR-Tools Routing Library.
Полный рабочий пример.
"""

import pandas as pd
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# ============================================================
# ДАННЫЕ
# ============================================================

demand_pairs = pd.DataFrame({
    'source_station_id': ['S1', 'S2', 'S3', 'S1', 'S4'],
    'target_station_id': ['S3', 'S4', 'S2', 'S4', 'S1'],
    'commodity_type':    ['ordinary', 'electric', 'ordinary', 'electric', 'ordinary'],
    'quantity':          [5, 3, 4, 2, 6],
})

trucks = pd.DataFrame({
    'resource_id': ['T1', 'T2'],
    'capacity':    [10, 8],
    'depot_home':  ['D1', 'D2'],
    'rate':        [2.0, 2.5],
})

distances = pd.DataFrame({
    'source_id':   ['D1','D1','D1','D1','D1','D1',
                    'D2','D2','D2','D2','D2','D2',
                    'S1','S1','S1','S1','S1','S1',
                    'S2','S2','S2','S2','S2','S2',
                    'S3','S3','S3','S3','S3','S3',
                    'S4','S4','S4','S4','S4','S4'],
    'source_type': ['depot']*6 + ['depot']*6 +
                   ['station']*6 + ['station']*6 +
                   ['station']*6 + ['station']*6,
    'target_id':   ['D1','D2','S1','S2','S3','S4'] * 6,
    'target_type': ['depot','depot','station','station','station','station'] * 6,
    'distance_km': [
        0, 10, 3, 7, 5, 8,
        10, 0, 8, 4, 6, 3,
        3, 8, 0, 6, 4, 7,
        7, 4, 6, 0, 5, 3,
        5, 6, 4, 5, 0, 6,
        8, 3, 7, 3, 6, 0,
    ],
    'duration_hr': [
        0.0, 0.5, 0.15, 0.35, 0.25, 0.4,
        0.5, 0.0, 0.4, 0.2, 0.3, 0.15,
        0.15, 0.4, 0.0, 0.3, 0.2, 0.35,
        0.35, 0.2, 0.3, 0.0, 0.25, 0.15,
        0.25, 0.3, 0.2, 0.25, 0.0, 0.3,
        0.4, 0.15, 0.35, 0.15, 0.3, 0.0,
    ],
})

# ============================================================
# КОНСТАНТЫ
# ============================================================

COST_SCALE = 1000      # float -> int для стоимости
TIME_SCALE = 3600      # часы -> секунды
MAX_TIME_HOURS = 2     # ограничение по времени маршрута
PENALTY_PER_BIKE = 100_000  # штраф за непервезённый велосипед
SOLVER_TIME_LIMIT = 5      # секунд CPU на решение

# ============================================================
# ШАГ 1: Модель узлов
# ============================================================

num_pairs = len(demand_pairs)
num_vehicles = len(trucks)

node_to_location = {}
for i in range(num_pairs):
    row = demand_pairs.iloc[i]
    node_to_location[2 * i]     = (row['source_station_id'], 'station')
    node_to_location[2 * i + 1] = (row['target_station_id'], 'station')

unique_depots = trucks['depot_home'].unique().tolist()
depot_node_start = 2 * num_pairs
depot_id_to_node = {}
for j, depot_id in enumerate(unique_depots):
    node_idx = depot_node_start + j
    node_to_location[node_idx] = (depot_id, 'depot')
    depot_id_to_node[depot_id] = node_idx

total_nodes = 2 * num_pairs + len(unique_depots)

# ============================================================
# ШАГ 2: Lookup-таблицы
# ============================================================

dist_lookup = {}
time_lookup = {}
for _, row in distances.iterrows():
    key = (row['source_id'], row['source_type'], row['target_id'], row['target_type'])
    dist_lookup[key] = row['distance_km']
    time_lookup[key] = row['duration_hr']


def get_distance(node_a, node_b):
    loc_a = node_to_location[node_a]
    loc_b = node_to_location[node_b]
    return dist_lookup.get((loc_a[0], loc_a[1], loc_b[0], loc_b[1]), 0)


def get_duration(node_a, node_b):
    loc_a = node_to_location[node_a]
    loc_b = node_to_location[node_b]
    return time_lookup.get((loc_a[0], loc_a[1], loc_b[0], loc_b[1]), 0)

# ============================================================
# ШАГ 3: IndexManager + RoutingModel
# ============================================================

starts = [depot_id_to_node[trucks.iloc[v]['depot_home']] for v in range(num_vehicles)]
ends   = [depot_id_to_node[trucks.iloc[v]['depot_home']] for v in range(num_vehicles)]

manager = pywrapcp.RoutingIndexManager(total_nodes, num_vehicles, starts, ends)
routing = pywrapcp.RoutingModel(manager)

# ============================================================
# ШАГ 4: Cost callbacks
# ============================================================

for v in range(num_vehicles):
    rate = trucks.iloc[v]['rate']

    def make_cost_cb(r):
        def cb(from_index, to_index):
            fn = manager.IndexToNode(from_index)
            tn = manager.IndexToNode(to_index)
            return int(get_distance(fn, tn) * r * COST_SCALE)
        return cb

    cb_idx = routing.RegisterTransitCallback(make_cost_cb(rate))
    routing.SetArcCostEvaluatorOfVehicle(cb_idx, v)

# ============================================================
# ШАГ 5: Time dimension
# ============================================================

def time_callback(from_index, to_index):
    fn = manager.IndexToNode(from_index)
    tn = manager.IndexToNode(to_index)
    return int(get_duration(fn, tn) * TIME_SCALE)

time_cb_idx = routing.RegisterTransitCallback(time_callback)
routing.AddDimension(time_cb_idx, 0, MAX_TIME_HOURS * TIME_SCALE, True, 'Time')

# ============================================================
# ШАГ 6: Capacity dimension
# ============================================================

node_demands = [0] * total_nodes
for i in range(num_pairs):
    qty = int(demand_pairs.iloc[i]['quantity'])
    node_demands[2 * i]     = qty
    node_demands[2 * i + 1] = -qty


def demand_callback(from_index):
    return int(node_demands[manager.IndexToNode(from_index)])

demand_cb_idx = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_cb_idx, 0, [int(c) for c in trucks['capacity'].tolist()], True, 'Capacity'
)

# ============================================================
# ШАГ 7: Pickup & Delivery
# ============================================================

time_dim = routing.GetDimensionOrDie('Time')

for i in range(num_pairs):
    p_idx = manager.NodeToIndex(2 * i)
    d_idx = manager.NodeToIndex(2 * i + 1)

    routing.AddPickupAndDelivery(p_idx, d_idx)
    routing.solver().Add(routing.VehicleVar(p_idx) == routing.VehicleVar(d_idx))
    routing.solver().Add(time_dim.CumulVar(p_idx) <= time_dim.CumulVar(d_idx))

# ============================================================
# ШАГ 8: Disjunctions (опциональные пары)
# ============================================================

for i in range(num_pairs):
    p_idx = manager.NodeToIndex(2 * i)
    d_idx = manager.NodeToIndex(2 * i + 1)
    penalty = int(demand_pairs.iloc[i]['quantity']) * PENALTY_PER_BIKE
    routing.AddDisjunction([p_idx, d_idx], int(penalty), 2)

# ============================================================
# ШАГ 9: Запуск
# ============================================================

search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.FromSeconds(SOLVER_TIME_LIMIT)

solution = routing.SolveWithParameters(search_params)

# ============================================================
# ШАГ 10: Вывод результатов
# ============================================================

if not solution:
    print("Решение не найдено!")
else:
    print("=== РЕЗУЛЬТАТЫ ===\n")

    for v in range(num_vehicles):
        truck = trucks.iloc[v]
        print(f"Грузовик {truck['resource_id']} "
              f"(депо {truck['depot_home']}, "
              f"capacity={truck['capacity']}, "
              f"rate={truck['rate']} USD/km)")

        index = routing.Start(v)
        route_dist = 0
        prev_node = None

        while True:
            node = manager.IndexToNode(index)
            loc_id, loc_type = node_to_location[node]
            t = solution.Value(time_dim.CumulVar(index)) / TIME_SCALE

            # Определяем, что за узел
            label = f"{loc_type}:{loc_id}"
            if node < 2 * num_pairs:
                pair_idx = node // 2
                is_pickup = (node % 2 == 0)
                pair = demand_pairs.iloc[pair_idx]
                action = "PICKUP" if is_pickup else "DELIVER"
                label += (f" [{action} {pair['quantity']} "
                          f"{pair['commodity_type']}]")

            if prev_node is not None:
                seg_dist = get_distance(prev_node, node)
                route_dist += seg_dist

            print(f"  t={t:.2f}ч  {label}")

            if routing.IsEnd(index):
                break
            prev_node = node
            index = solution.Value(routing.NextVar(index))

        route_cost = route_dist * truck['rate']
        print(f"  Пробег: {route_dist:.1f} км, "
              f"стоимость: ${route_cost:.2f}\n")

    # Итоги
    bikes_moved = 0
    bikes_dropped = 0
    for i in range(num_pairs):
        p_idx = manager.NodeToIndex(2 * i)
        if not routing.IsStart(solution.Value(routing.NextVar(p_idx))):
            bikes_moved += demand_pairs.iloc[i]['quantity']
        else:
            qty = demand_pairs.iloc[i]['quantity']
            bikes_dropped += qty
            row = demand_pairs.iloc[i]
            print(f"ПРОПУЩЕНО: {row['source_station_id']} -> "
                  f"{row['target_station_id']}, "
                  f"{qty} {row['commodity_type']}")

    total_qty = demand_pairs['quantity'].sum()
    print(f"\nИтого перевезено: {bikes_moved} / {total_qty} велосипедов")
    print(f"Objective value (масштаб.): {solution.ObjectiveValue()}")

=== РЕЗУЛЬТАТЫ ===

Грузовик T1 (депо D1, capacity=10, rate=2.0 USD/km)
  t=0.00ч  depot:D1
  t=0.15ч  station:S1 [PICKUP 5 ordinary]
  t=0.15ч  station:S1 [PICKUP 2 electric]
  t=0.35ч  station:S3 [DELIVER 5 ordinary]
  t=0.35ч  station:S3 [PICKUP 4 ordinary]
  t=0.60ч  station:S2 [PICKUP 3 electric]
  t=0.60ч  station:S2 [DELIVER 4 ordinary]
  t=0.75ч  station:S4 [DELIVER 3 electric]
  t=0.75ч  station:S4 [DELIVER 2 electric]
  t=0.75ч  station:S4 [PICKUP 6 ordinary]
  t=1.10ч  station:S1 [DELIVER 6 ordinary]
  t=1.25ч  depot:D1
  Пробег: 25.0 км, стоимость: $50.00

Грузовик T2 (депо D2, capacity=8, rate=2.5 USD/km)
  t=0.00ч  depot:D2
  t=0.00ч  depot:D2
  Пробег: 0.0 км, стоимость: $0.00


Итого перевезено: 20 / 20 велосипедов
Objective value (масштаб.): 50000
